## Imports

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [3]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [4]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [38]:
## Parameter 
start_date = '20240820'
end_date = '20240929'

In [39]:
sample_category = ['control']
city_name = ['Hyderabad'] #'Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Jaipur', 'Mumbai'

In [40]:
def query(city, category):
    
    base_query = f"""
    
        WITH base AS (
        
            SELECT 
                yyyymmdd,
                sample_category,
                customer_id
            FROM
                iceberg.experiments_internal.rapidex_user_customer_adslots_v1_root
            WHERE
                experiment_name = 'Ads-Holdout-Group-Experiment' 
                AND execution_name = 'holdout_segment_hyd'
                AND sample_category = '{category}'
                AND yyyymmdd >= '20240806'
                AND yyyymmdd <= '20240929'
            ),
            
            t1 AS (
        
            SELECT 
                DISTINCT ols.customer_id,
                DATE_FORMAT(DATE_TRUNC('week', DATE_PARSE(ols.yyyymmdd, '%Y%m%d')),'%Y-%m-%d') as date_period1
            FROM 
                orders.order_logs_snapshot ols
            JOIN 
                base
                ON base.customer_id = ols.customer_id
            WHERE
                ols.yyyymmdd >= '{start_date}'
                AND ols.yyyymmdd <= '{end_date}'
                AND city_name = '{city}' 
                AND service_obj_service_name in ( 'Auto', 'Auto Pool', 'Auto NCR', 'Auto NCR', 'Bike Pink', 'Link', 'Bike Lite', 'Bike Metro', 'CabEconomy', 'CabPremium' )
                AND order_status = 'dropped'
            ),
            
            t2 AS (
        
            SELECT 
                DISTINCT ols.customer_id,
                DATE_FORMAT(DATE_TRUNC('week', DATE_PARSE(ols.yyyymmdd, '%Y%m%d')),'%Y-%m-%d') as date_period2
            FROM 
                orders.order_logs_snapshot ols
            JOIN 
                base
                ON base.customer_id = ols.customer_id
            WHERE
                ols.yyyymmdd >= '{start_date}'
                AND ols.yyyymmdd <= '{end_date}'
                AND city_name = '{city}' 
                AND service_obj_service_name in ( 'Auto', 'Auto Pool', 'Auto NCR', 'Auto NCR', 'Bike Pink', 'Link', 'Bike Lite', 'Bike Metro', 'CabEconomy', 'CabPremium' )
                AND order_status = 'dropped'
            ),
            
            t3 AS (
            
            SELECT 
                date_period1 date_period3,
                COUNT(DISTINCT customer_id) as total_cx
            FROM 
                t1
            GROUP BY 1
            ),
            
            t4 AS (
            SELECT 
                date_period1,
                date_period2,
                COUNT(DISTINCT t1.customer_id) as count_t1,
                COUNT(DISTINCT t2.customer_id) as count_t2 
            FROM 
                t1
            LEFT JOIN 
                t2 
                ON t1.customer_id = t2.customer_id 
                AND date_period1 <= date_period2
            GROUP BY 1,2
            )
            
        SELECT 
            date_period1 start_date, 
            date_period2, 
            CASE WHEN date_period1 = date_period2 THEN 100.00 ELSE (100.00*(t4.count_t1))/total_cx END AS count_2
        FROM 
            t4 
        JOIN 
            t3 
            ON t3.date_period3 = t4.date_period1
        WHERE 
            date_period2 IS NOT NULL
        ORDER BY 2,1
    
    """
    
    df_base_query = pd.read_sql(base_query, connection)
    
    return df_base_query

In [41]:
# List to store names of created DataFrames
df_names = []

for city in city_name:
    for category in sample_category:
        df_name = f"{city}_{category}"
        print(df_name)
        
        # Call the query function
        df = query(city, category)
        
        # Add city and category columns
        df['city'] = city
        df['category'] = category
        
        # Store the DataFrame with a unique name
        globals()[df_name] = df

        # Add the DataFrame name to the list
        df_names.append(df_name)

# Print the names of all created DataFrames
print("DataFrames created:", df_names)

Hyderabad_control


DatabaseError: {'message': 'Query exceeded the maximum execution time limit of 10.00m', 'errorCode': 131075, 'errorName': 'EXCEEDED_TIME_LIMIT', 'errorType': 'INSUFFICIENT_RESOURCES', 'failureInfo': {'type': 'io.trino.spi.TrinoException', 'message': 'Query exceeded the maximum execution time limit of 10.00m', 'suppressed': [], 'stack': ['io.trino.execution.QueryTracker.enforceTimeLimits(QueryTracker.java:187)', 'io.trino.execution.QueryTracker.lambda$start$0(QueryTracker.java:89)', 'com.google.common.util.concurrent.MoreExecutors$ScheduledListeningDecorator$NeverSuccessfulListenableFutureTask.run(MoreExecutors.java:730)', 'java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)', 'java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)', 'java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)', 'java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.jav

In [43]:
# Hyderabad_control.to_csv('Hyderabad_control.csv', index=False)

In [9]:
Bangalore_test.to_csv('Bangalore_test.csv', index=False)
Bangalore_control.to_csv('Bangalore_control.csv', index=False)

Chennai_test.to_csv('Chennai_test.csv', index=False)
Chennai_control.to_csv('Chennai_control.csv', index=False)

Delhi_test.to_csv('Delhi_test.csv', index=False)
Delhi_control.to_csv('Delhi_control.csv', index=False)

Kolkata_test.to_csv('Kolkata_test.csv', index=False)
Kolkata_control.to_csv('Kolkata_control.csv', index=False)

Jaipur_test.to_csv('Jaipur_test.csv', index=False)
Jaipur_control.to_csv('Jaipur_control.csv', index=False)

Mumbai_test.to_csv('Mumbai_test.csv', index=False)
Mumbai_control.to_csv('Mumbai_control.csv', index=False)

Hyderabad_test.to_csv('Hyderabad_test.csv', index=False)


In [44]:
Bangalore_test.head(2)

,start_date,date_period2,count_2,city,category
0,2024-08-05,2024-08-05,100.00,Bangalore,test
1,2024-08-05,2024-08-12,49.93,Bangalore,test


In [50]:
Bangalore_test.pivot(index ='start_date' , columns ='date_period2', values =['count_2']).fillna('')

count_2                                              \
date_period2 2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                            
2024-08-05        100.0      49.93      46.36      44.38      43.57   
2024-08-12                   100.0      48.06      43.89      43.33   
2024-08-19                              100.0      48.43      45.71   
2024-08-26                                         100.0       49.7   
2024-09-02                                                    100.0   
2024-09-09                                                            
2024-09-16                                                            
2024-09-23                                                            

                                               
date_period2 2024-09-09 2024-09-16 2024-09-23  
start_date                                     
2024-08-05        43.18      41.76      41.21  
2024-08-12        42.34      41.08      39.73  
2024-08-19        43.96       42.9      41.62  
2024-08-26        46.65      44.52      43.73  
2024-09-02        49.82      45.99      44.03  
2024-09-09        100.0      49.15      45.15  
2024-09-16                   100.0      49.40  
2024-09-23                             100.00

In [49]:
Bangalore_control.pivot(index ='start_date' , columns ='date_period2', values =['count_2']).fillna('')

count_2                                              \
date_period2 2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                            
2024-08-05        100.0      49.98      46.83      44.97      44.04   
2024-08-12                   100.0      48.25      44.21      43.08   
2024-08-19                              100.0      48.77      45.38   
2024-08-26                                         100.0      49.68   
2024-09-02                                                    100.0   
2024-09-09                                                            
2024-09-16                                                            
2024-09-23                                                            

                                               
date_period2 2024-09-09 2024-09-16 2024-09-23  
start_date                                     
2024-08-05        43.45      42.05      41.06  
2024-08-12        42.47      40.97      39.61  
2024-08-19        44.22       42.7      41.27  
2024-08-26        46.57      44.69      43.34  
2024-09-02        49.82      45.93      43.94  
2024-09-09        100.0      49.38      45.20  
2024-09-16                   100.0      49.42  
2024-09-23                             100.00

In [6]:
local_extract = '/Users/E2074/local-datasets/customer/ads-monetisation/holdout-group/experiment-result'

In [7]:
# Mapping dictionary
execution_city_name_mapping = {
    'holdout_segment_ban': 'Bangalore',
    'holdout_segment_che': 'Chennai',
    'holdout_segment_del': 'Delhi',
    'holdout_segment_hyd': 'Hyderabad',
    'holdout_segment_jai': 'Jaipur',
    'holdout_segment_kol': 'Kolkata',
    'holdout_segment_mum': 'Mumbai'
}

In [8]:
df_overall_data = pd.read_csv(local_extract + '/city_level_experiment_retention_data.csv')
print(df_overall_data.shape)
df_overall_data['city_name'] = df_overall_data['execution_name'].map(execution_city_name_mapping)
df_overall_data.head(5)

(984, 5)


,execution_name,sample_category,start_date,retention_date,retention_percentage,city_name
0,holdout_segment_ban,control,2024-08-05,2024-08-05,100.00,Bangalore
1,holdout_segment_ban,control,2024-08-05,2024-08-12,51.73,Bangalore
2,holdout_segment_ban,control,2024-08-05,2024-08-19,48.67,Bangalore
3,holdout_segment_ban,control,2024-08-05,2024-08-26,46.56,Bangalore
4,holdout_segment_ban,control,2024-08-05,2024-09-02,45.70,Bangalore


In [9]:
def get_city_retention_cohort(df, city_name):
    
    df_control = df[(df['city_name'] == city_name) & (df['sample_category'] == 'control')]
    df_test = df[(df['city_name'] == city_name) & (df['sample_category'] == 'test')]
    df_merge = pd.merge(df_control,df_test, how='inner',on=['city_name', 'start_date', 'retention_date'])
    df_merge['retention_delta'] =  df_merge['retention_percentage_x'] - df_merge['retention_percentage_y']
    df_merge['difference'] = 'delta - (control - test)'
    df1 = df_control.pivot(index=['start_date'], 
                           columns=['city_name', 'sample_category', 'retention_date'],
                           values='retention_percentage').fillna('')
    df2 = df_test.pivot(index=['start_date'], 
                        columns=['city_name', 'sample_category', 'retention_date'],
                        values='retention_percentage').fillna('')
    df3 = df_merge.pivot(index=['start_date'], 
                         columns=['city_name', 'difference', 'retention_date'], 
                         values='retention_delta').fillna('')
    print('standard deviation → ', round(df_merge['retention_delta'].std(), 2))
    
    return display(df1,df2,df3)

In [16]:
get_city_retention_cohort(df_overall_data, 'Mumbai')

standard deviation →  0.74


city_name           Mumbai                                              \
sample_category    control                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      45.32      43.24      40.76      40.82   
2024-08-12                      100.0      44.76      40.32      40.14   
2024-08-19                                 100.0       41.9      40.22   
2024-08-26                                            100.0      45.62   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           39.18      36.64      38.15      36.23      36.98   
2024-08-12           38.51      36.02      37.01      35.30      36.17   
2024-08-19           38.32      35.48      36.57      34.60      35.44   
2024-08-26           41.71      38.69      39.89      37.62      38.04   
2024-09-02            43.6      38.42      39.18      37.01      37.68   
2024-09-09           100.0      41.27      40.08      37.44      38.15   
2024-09-16                      100.0      46.00      41.74      41.46   
2024-09-23                                100.00      43.04      41.62   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           36.21      35.91      32.58  
2024-08-12           35.34      34.79      32.30  
2024-08-19           34.66      34.31      31.75  
2024-08-26           37.24      36.82      33.25  
2024-09-02           36.54      35.92      33.04  
2024-09-09           36.86      36.11      33.13  
2024-09-16           40.47      39.37      35.37  
2024-09-23           40.25      38.99      34.63

city_name           Mumbai                                              \
sample_category       test                                               
retention_date  2024-08-05 2024-08-12 2024-08-19 2024-08-26 2024-09-02   
start_date                                                               
2024-08-05           100.0      45.99      44.09      41.12      41.24   
2024-08-12                      100.0      45.96       41.4       40.3   
2024-08-19                                 100.0       42.8      40.95   
2024-08-26                                            100.0       45.7   
2024-09-02                                                       100.0   
2024-09-09                                                               
2024-09-16                                                               
2024-09-23                                                               

city_name                                                               \
sample_category                                                          
retention_date  2024-09-09 2024-09-16 2024-09-23 2024-09-30 2024-10-07   
start_date                                                               
2024-08-05           40.55      38.19      39.51      37.58      37.96   
2024-08-12           39.53       36.9      38.82      37.39      37.02   
2024-08-19           39.67      36.86      38.01      36.47      36.56   
2024-08-26           42.94      39.65      40.63      39.10      38.39   
2024-09-02           44.49      38.88      39.24      39.08      37.80   
2024-09-09           100.0      41.65      41.37      39.68      38.41   
2024-09-16                      100.0      47.58      43.90      40.80   
2024-09-23                                100.00      45.01      41.92   

city_name                                         
sample_category                                   
retention_date  2024-10-14 2024-10-21 2024-10-28  
start_date                                        
2024-08-05           37.61      35.08      32.55  
2024-08-12           36.50      34.90      32.44  
2024-08-19           36.44      34.81      33.38  
2024-08-26           38.12      36.99      33.80  
2024-09-02           37.64      35.78      33.29  
2024-09-09           38.10      35.98      33.70  
2024-09-16           41.19      38.35      35.25  
2024-09-23           40.58      38.57      34.68

city_name                        Mumbai                                   \
difference     delta - (control - test)                                    
retention_date               2024-08-05 2024-08-12 2024-08-19 2024-08-26   
start_date                                                                 
2024-08-05                          0.0      -0.67      -0.85      -0.36   
2024-08-12                                     0.0       -1.2      -1.08   
2024-08-19                                                0.0       -0.9   
2024-08-26                                                           0.0   
2024-09-02                                                                 
2024-09-09                                                                 
2024-09-16                                                                 
2024-09-23                                                                 

city_name                                                              \
difference                                                              
retention_date 2024-09-02 2024-09-09 2024-09-16 2024-09-23 2024-09-30   
start_date                                                              
2024-08-05          -0.42      -1.37      -1.55      -1.36      -1.35   
2024-08-12          -0.16      -1.02      -0.88      -1.81      -2.09   
2024-08-19          -0.73      -1.35      -1.38      -1.44      -1.87   
2024-08-26          -0.08      -1.23      -0.96      -0.74      -1.48   
2024-09-02            0.0      -0.89      -0.46      -0.06      -2.07   
2024-09-09                       0.0      -0.38      -1.29      -2.24   
2024-09-16                                  0.0      -1.58      -2.16   
2024-09-23                                            0.00      -1.97   

city_name                                                   
difference                                                  
retention_date 2024-10-07 2024-10-14 2024-10-21 2024-10-28  
start_date                                                  
2024-08-05          -0.98      -1.40       0.83       0.03  
2024-08-12          -0.85      -1.16      -0.11      -0.14  
2024-08-19          -1.12      -1.78      -0.50      -1.63  
2024-08-26          -0.35      -0.88      -0.17      -0.55  
2024-09-02          -0.12      -1.10       0.14      -0.25  
2024-09-09          -0.26      -1.24       0.13      -0.57  
2024-09-16           0.66      -0.72       1.02       0.12  
2024-09-23          -0.30      -0.33       0.42      -0.05